# 무슨 옷과 무슨 색?

이번 실습에서는 의류 이미지를 보고 옷의 종류와 색상을 함께 예측하는 문제를 다룹니다. 하나의 이미지가 `blue`이면서 동시에 `shirt`일 수 있으므로, 이 문제는 하나의 정답만 고르는 분류가 아니라 여러 레이블을 함께 예측하는 다중 레이블 분류 문제입니다.

## 사용할 레이블

색상 레이블은 `black`, `blue`, `brown`, `green`, `red`, `white`이고, 의류 종류 레이블은 `dress`, `shirt`, `pants`, `shorts`, `shoes`입니다. 따라서 각 이미지는 총 11개의 라벨 컬럼으로 표현할 수 있습니다.

In [1]:
COLOR_LABELS = ['black', 'blue', 'brown', 'green', 'red', 'white']
CATEGORY_LABELS = ['dress', 'shirt', 'pants', 'shorts', 'shoes']
MULTI_LABEL_COLUMNS = COLOR_LABELS + CATEGORY_LABELS

print('색상 레이블:', COLOR_LABELS)
print('의류 종류 레이블:', CATEGORY_LABELS)
print('전체 레이블 수:', len(MULTI_LABEL_COLUMNS))

색상 레이블: ['black', 'blue', 'brown', 'green', 'red', 'white']
의류 종류 레이블: ['dress', 'shirt', 'pants', 'shorts', 'shoes']
전체 레이블 수: 11


## 데이터 준비 흐름

이번 노트북에서는 폴더 구조를 확인하고, 폴더 이름에서 라벨을 읽어 이미지 1장을 레코드 1개로 바꾼 뒤, 전체 이미지를 DataFrame으로 정리합니다. 이후 `color_index`를 추가한 상태로 train, validation, test 데이터로 나누고 CSV 파일로 저장합니다.

## 데이터셋 살펴보기

이미지는 `clothes_dataset` 아래에 폴더별로 정리되어 있고, 각 폴더 이름은 `blue_shirt`, `red_dress`처럼 색상과 의류 종류를 함께 담고 있습니다. 따라서 상위 폴더 이름만 읽어도 정답 레이블을 만들 수 있습니다.

In [2]:
from pathlib import Path

DATASET_DIR = Path('./clothes_dataset')
DATASET_DIR

WindowsPath('clothes_dataset')

In [3]:
folder_names = sorted([path.name for path in DATASET_DIR.iterdir() if path.is_dir()])

print('폴더 개수:', len(folder_names))
print('폴더 목록:')
for folder_name in folder_names:
    print('-', folder_name)

폴더 개수: 24
폴더 목록:
- black_dress
- black_pants
- black_shirt
- black_shoes
- black_shorts
- blue_dress
- blue_pants
- blue_shirt
- blue_shoes
- blue_shorts
- brown_pants
- brown_shoes
- brown_shorts
- green_pants
- green_shirt
- green_shoes
- green_shorts
- red_dress
- red_pants
- red_shoes
- white_dress
- white_pants
- white_shoes
- white_shorts


In [4]:
sample_folder = folder_names[0]
color, category = sample_folder.split('_', maxsplit=1)

print('예시 폴더명:', sample_folder)
print('색상:', color)
print('의류 종류:', category)

예시 폴더명: black_dress
색상: black
의류 종류: dress


## 데이터 처리 함수를 분리하기

데이터 준비 로직은 이후 다른 노트북에서도 재사용할 수 있도록 `utils/data_utils.py`에 분리합니다.

## 사전 준비

아직 `polars`가 설치되지 않았다면 먼저 설치합니다.

`pip install polars pyarrow`

In [5]:
from utils.data_utils import parse_folder_name

sample_folder = 'blue_shirt'
color, category = parse_folder_name(sample_folder)

print('폴더명:', sample_folder)
print('색상:', color)
print('의류 종류:', category)

폴더명: blue_shirt
색상: blue
의류 종류: shirt


## 이미지 1장을 레코드 1개로 바꾸기

이미지 파일 1장은 학습 데이터의 1행으로 바꿀 수 있습니다. 이 레코드에는 이미지 경로와 11개의 멀티라벨 값이 들어가며, 해당하는 색상과 의류 종류만 1이 됩니다.

In [6]:
from utils.data_utils import build_label_record

sample_image_path = next(DATASET_DIR.glob('*/*.jpg'))
sample_image_path

WindowsPath('clothes_dataset/black_dress/0097960878307e559459d98c9f9eaeeea0db1f94.jpg')

In [7]:
sample_record = build_label_record(sample_image_path)
sample_record

{'image': 'clothes_dataset\\black_dress\\0097960878307e559459d98c9f9eaeeea0db1f94.jpg',
 'black': 1,
 'blue': 0,
 'brown': 0,
 'green': 0,
 'red': 0,
 'white': 0,
 'dress': 1,
 'shirt': 0,
 'pants': 0,
 'shorts': 0,
 'shoes': 0,
 'color_index': 0}

## 전체 이미지를 DataFrame으로 만들기

이제 이미지 1장을 레코드 1개로 바꾸는 규칙을 전체 이미지에 반복합니다. 이번 실습에서는 데이터프레임 처리에 Polars를 사용하고, 이후 확장을 위해 `color_index` 컬럼도 함께 만듭니다.

In [8]:
from utils.data_utils import collect_image_paths, build_dataframe

image_paths = collect_image_paths(DATASET_DIR)

print('이미지 개수:', len(image_paths))
print('첫 번째 이미지 경로:', image_paths[0])

이미지 개수: 11385
첫 번째 이미지 경로: clothes_dataset\black_dress\0097960878307e559459d98c9f9eaeeea0db1f94.jpg


In [9]:
df = build_dataframe(DATASET_DIR)
df.head()

image,black,blue,brown,green,red,white,dress,shirt,pants,shorts,shoes,color_index
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""clothes_dataset\black_dress\00…",1,0,0,0,0,0,1,0,0,0,0,0
"""clothes_dataset\black_dress\03…",1,0,0,0,0,0,1,0,0,0,0,0
"""clothes_dataset\black_dress\05…",1,0,0,0,0,0,1,0,0,0,0,0
"""clothes_dataset\black_dress\05…",1,0,0,0,0,0,1,0,0,0,0,0
"""clothes_dataset\black_dress\06…",1,0,0,0,0,0,1,0,0,0,0,0


## 자료형과 컬럼 확인

라벨 컬럼과 `color_index`는 모두 작은 정수 범위만 가지므로 `UInt8`로 저장합니다. 이 단계에서는 데이터가 잘 만들어졌는지뿐 아니라 컬럼 구성과 자료형도 함께 확인합니다.

In [10]:
print(df.shape)
print(df.columns)
df.schema

(11385, 13)
['image', 'black', 'blue', 'brown', 'green', 'red', 'white', 'dress', 'shirt', 'pants', 'shorts', 'shoes', 'color_index']


Schema([('image', String),
        ('black', UInt8),
        ('blue', UInt8),
        ('brown', UInt8),
        ('green', UInt8),
        ('red', UInt8),
        ('white', UInt8),
        ('dress', UInt8),
        ('shirt', UInt8),
        ('pants', UInt8),
        ('shorts', UInt8),
        ('shoes', UInt8),
        ('color_index', UInt8)])

In [11]:
df.select(['image', 'black', 'dress', 'color_index']).head()

image,black,dress,color_index
str,u8,u8,u8
"""clothes_dataset\black_dress\00…",1,1,0
"""clothes_dataset\black_dress\03…",1,1,0
"""clothes_dataset\black_dress\05…",1,1,0
"""clothes_dataset\black_dress\05…",1,1,0
"""clothes_dataset\black_dress\06…",1,1,0


## train, validation, test로 분할하기

모델 학습에는 train 데이터, 학습 중 점검에는 validation 데이터, 마지막 평가는 test 데이터를 사용합니다. 이번 예제에서는 먼저 train과 test로 나눈 뒤, train을 다시 train과 validation으로 나눕니다.

In [12]:
from utils.data_utils import split_dataframe

train_df, val_df, test_df = split_dataframe(df)

print('전체 데이터 크기:', df.shape)
print('train 크기:', train_df.shape)
print('validation 크기:', val_df.shape)
print('test 크기:', test_df.shape)

전체 데이터 크기: (11385, 13)
train 크기: (5578, 13)
validation 크기: (2391, 13)
test 크기: (3416, 13)


In [13]:
train_df.head()

image,black,blue,brown,green,red,white,dress,shirt,pants,shorts,shoes,color_index
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""clothes_dataset\black_shirt\b5…",1,0,0,0,0,0,0,1,0,0,0,0
"""clothes_dataset\blue_shirt\4b5…",0,1,0,0,0,0,0,1,0,0,0,1
"""clothes_dataset\black_shoes\2e…",1,0,0,0,0,0,0,0,0,0,1,0
"""clothes_dataset\blue_dress\fa3…",0,1,0,0,0,0,1,0,0,0,0,1
"""clothes_dataset\black_pants\82…",1,0,0,0,0,0,0,0,1,0,0,0


In [14]:
val_df.head()

image,black,blue,brown,green,red,white,dress,shirt,pants,shorts,shoes,color_index
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""clothes_dataset\green_shoes\3b…",0,0,0,1,0,0,0,0,0,0,1,3
"""clothes_dataset\red_shoes\358f…",0,0,0,0,1,0,0,0,0,0,1,4
"""clothes_dataset\black_shirt\19…",1,0,0,0,0,0,0,1,0,0,0,0
"""clothes_dataset\black_shoes\f6…",1,0,0,0,0,0,0,0,0,0,1,0
"""clothes_dataset\red_dress\fee2…",0,0,0,0,1,0,1,0,0,0,0,4


In [15]:
test_df.head()

image,black,blue,brown,green,red,white,dress,shirt,pants,shorts,shoes,color_index
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""clothes_dataset\blue_shirt\a69…",0,1,0,0,0,0,0,1,0,0,0,1
"""clothes_dataset\white_dress\89…",0,0,0,0,0,1,1,0,0,0,0,5
"""clothes_dataset\black_pants\4d…",1,0,0,0,0,0,0,0,1,0,0,0
"""clothes_dataset\blue_shoes\e3f…",0,1,0,0,0,0,0,0,0,0,1,1
"""clothes_dataset\white_shoes\ee…",0,0,0,0,0,1,0,0,0,0,1,5


## CSV 파일로 저장하기

학습 노트북에서 바로 사용할 수 있도록 분할된 데이터를 CSV 파일로 저장합니다. `nocolorinfo` 폴더에는 기본 멀티라벨 CSV를 저장하고, `colorinfo` 폴더에는 `color_index`가 포함된 CSV를 저장합니다.

In [16]:
from utils.data_utils import save_split_dataframes

save_split_dataframes(train_df, val_df, test_df, output_dir='./csv_data')
print('CSV 저장 완료')

CSV 저장 완료


In [17]:
sorted(str(path) for path in Path('./csv_data').rglob('*.csv'))

['csv_data\\colorinfo\\.ipynb_checkpoints\\test_color-checkpoint.csv',
 'csv_data\\colorinfo\\test_color.csv',
 'csv_data\\colorinfo\\train_color.csv',
 'csv_data\\colorinfo\\val_color.csv',
 'csv_data\\nocolorinfo\\test.csv',
 'csv_data\\nocolorinfo\\train.csv',
 'csv_data\\nocolorinfo\\val.csv']

## 불러온 결과 확인하기

이 단계에서는 저장된 CSV가 정상적으로 다시 읽히는지 확인합니다. 이후 학습 노트북에서는 이미지 폴더를 다시 탐색하지 않고, 이 `load_split_dataframes()` 함수를 이용해 필요한 데이터만 바로 불러와 사용할 수 있습니다.

In [18]:
from utils.data_utils import load_split_dataframes

In [19]:
loaded_train_df, loaded_val_df, loaded_test_df = load_split_dataframes(
    input_dir='./csv_data',
    use_color_info=False
)

In [20]:
print('train 크기:', loaded_train_df.shape)
print('validation 크기:', loaded_val_df.shape)
print('test 크기:', loaded_test_df.shape)

train 크기: (5578, 12)
validation 크기: (2391, 12)
test 크기: (3416, 12)


In [21]:
loaded_train_color_df, loaded_val_color_df, loaded_test_color_df = load_split_dataframes(
    input_dir='./csv_data',
    use_color_info=True
)

In [22]:
loaded_train_df.head()

image,black,blue,brown,green,red,white,dress,shirt,pants,shorts,shoes
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""clothes_dataset\black_shirt\b5…",1,0,0,0,0,0,0,1,0,0,0
"""clothes_dataset\blue_shirt\4b5…",0,1,0,0,0,0,0,1,0,0,0
"""clothes_dataset\black_shoes\2e…",1,0,0,0,0,0,0,0,0,0,1
"""clothes_dataset\blue_dress\fa3…",0,1,0,0,0,0,1,0,0,0,0
"""clothes_dataset\black_pants\82…",1,0,0,0,0,0,0,0,1,0,0


In [23]:
print('train_color 크기:', loaded_train_color_df.shape)
print('validation_color 크기:', loaded_val_color_df.shape)
print('test_color 크기:', loaded_test_color_df.shape)

train_color 크기: (5578, 13)
validation_color 크기: (2391, 13)
test_color 크기: (3416, 13)


In [24]:
loaded_train_color_df.head()

image,black,blue,brown,green,red,white,dress,shirt,pants,shorts,shoes,color_index
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""clothes_dataset\black_shirt\b5…",1,0,0,0,0,0,0,1,0,0,0,0
"""clothes_dataset\blue_shirt\4b5…",0,1,0,0,0,0,0,1,0,0,0,1
"""clothes_dataset\black_shoes\2e…",1,0,0,0,0,0,0,0,0,0,1,0
"""clothes_dataset\blue_dress\fa3…",0,1,0,0,0,0,1,0,0,0,0,1
"""clothes_dataset\black_pants\82…",1,0,0,0,0,0,0,0,1,0,0,0
